# 🤖 MLflow 3-Class Churn Model Training V2 (Advanced MLOps with FLAML & SHAP)

สมุดโน้ตเล่มนี้ประกอบไปด้วยระบบ **MLOps Pipeline** เต็มรูปแบบ:
1. **Auto-dependency Installation**: ตรวจสอบและติดตั้งไลบรารีที่จำเป็น (FLAML, SHAP) อัตโนมัติ
2. **Data Ingestion & Versioning**: ดึงข้อมูลจาก BigQuery และล็อกคีย์แฮชตารางเวอร์ชั่นข้อมูลด้วย MLflow Dataset Tracking
3. **Class Weight Adjustment**: แก้ปัญหาข้อมูลไม่สมดุล (Class Imbalance) เพื่อเพิ่มประสิทธิภาพโมเดล
4. **FLAML AutoML with K-Fold Cross Validation**: สุ่มหาโมเดลและการตั้งค่าที่ดีที่สุดแบบอัตโนมัติภายในเวลาจำกัด พร้อมระบบบันทึกรันย่อย (Nested Runs)
5. **SHAP Explainable AI**: บันทึกคำอธิบายการตัดสินใจของฟีเจอร์ต่างๆ แบบละเอียดยิบส่งตรงเข้าเซิร์ฟเวอร์ MLflow
6. **Model Registry**: ขึ้นทะเบียนเวอร์ชั่นโมเดลลงในคลังกลางเพื่อนำไปสแตนด์บายใช้งานจริง

> **V2 Changes**: รวม 4 ฟีเจอร์ที่ซ้ำซ้อน (`is_low_score`, `first_purchase_bad_review`, `first_purchase_late`, `is_extremely_late`) เป็นฟีเจอร์เดียว `bad_experience_score` (0-4) เพื่อลดความซ้ำซ้อนและเพิ่มสัญญาณความลึก

---

### 🛠️ 0. Auto-dependency Installation
ตรวจสอบความพร้อมและติดตั้งไลบรารีเพิ่มเติมหากตรวจไม่พบใน Virtual Environment ของคุณ

In [1]:
import sys
import subprocess

required_libs = ["flaml", "shap", "matplotlib", "seaborn"]
installed_libs = []

for lib in required_libs:
    try:
        __import__(lib)
    except ImportError:
        print(f"📦 Installing {lib}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", lib])
        installed_libs.append(lib)

print(f"✅ All libraries are ready! Installed in this session: {installed_libs if installed_libs else 'None'}")

d:\Desktop\Churn interi Dashboard\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ All libraries are ready! Installed in this session: None


### 📦 1. Import Libraries & โหลดข้อมูลจาก BigQuery

In [2]:
import os
import numpy as np
import pandas as pd
from google.oauth2 import service_account
from google.cloud import bigquery

# โหลดกุญแจเชื่อมต่อ BigQuery
key_path = "gcp-key.json"
credentials = service_account.Credentials.from_service_account_file(key_path)
client = bigquery.Client(credentials=credentials, project=credentials.project_id)

print("📥 Querying data from BigQuery...")
query = """
    SELECT * 
    FROM `academic-moon-483615-t2.analytics_olist.mart_churn_features` 
    WHERE split = 'train'
"""
df_raw = client.query(query).to_dataframe()
print(f"✅ Load Complete! Data Shape: {df_raw.shape}")
df_raw.head(3)

📥 Querying data from BigQuery...
✅ Load Complete! Data Shape: (70575, 15)


,order_id,customer_unique_id,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,price,freight_value,payment_installments,payment_value,payment_sequential,uses_voucher,review_score,product_category_name,split
0,2e287cee637969033ec9ab49602e61ef,2a80d0e64a7056796f27b4f0cb909182,unavailable,2017-01-30 15:48:53+00:00,NaT,2017-03-10 00:00:00+00:00,0E-9,0E-9,1,128.000000000,1,0,3.0,None,train
1,e58b8168894c93cf61d36c90775acd85,886c05aaa3a617c8b3a6b441eb88de2f,unavailable,2017-02-26 16:13:36+00:00,NaT,2017-03-22 00:00:00+00:00,0E-9,0E-9,4,87.040000000,1,0,1.0,None,train
2,cfbf015b7e4efd77c1a2ee7e337aea60,e677e3ecf81af42f9bb403785fff0904,unavailable,2017-11-07 17:12:46+00:00,NaT,2017-11-30 00:00:00+00:00,0E-9,0E-9,1,68.040000000,1,0,1.0,None,train


### 🏷️ 2. สร้าง Target Label 3 Classes & คำนวณ Class Weights
แบ่งกลุ่มลูกค้าเป็น 3 คลาส และเตรียมค่าน้ำหนักเพื่อปรับแก้โครงสร้างข้อมูลไม่สมดุล

In [3]:
from sklearn.utils.class_weight import compute_class_weight

# จัดเรียงประวัติตามลูกค้าและเวลาซื้อ
df = df_raw.sort_values(["customer_unique_id", "order_purchase_timestamp"]).reset_index(drop=True)

# คำนวณวันซื้อครั้งถัดไป
df["next_purchase_date"] = df.groupby("customer_unique_id")["order_purchase_timestamp"].shift(-1)
df["days_to_next"] = (df["next_purchase_date"] - df["order_purchase_timestamp"]).dt.days

# ฟังก์ชันแบ่ง 3 Class
def define_3classes(row):
    days = row["days_to_next"]
    if pd.notna(days):
        if days <= 180:
            return 0  # Class 0: Stay
        else:
            return 1  # Class 1: Delayed Return
    else:
        return 2      # Class 2: True Churn

df["label_3class"] = df.apply(define_3classes, axis=1)

counts = df["label_3class"].value_counts().sort_index()
pcts = df["label_3class"].value_counts(normalize=True).sort_index() * 100

class_names = {
    0: "Class 0: Stay (<= 180 days)",
    1: "Class 1: Delayed Return (> 180 days)",
    2: "Class 2: True Churn (No Return)"
}

print("📊 --- 3-Class Distribution ---")
for c_id, name in class_names.items():
    print(f"{name:<45} | Count: {counts.get(c_id, 0):>6,} | Ratio: {pcts.get(c_id, 0):.2f}%")

# คำนวณน้ำหนักแต่ละ Class (Class Weights)
classes = np.unique(df["label_3class"])
weights = compute_class_weight(class_weight="balanced", classes=classes, y=df["label_3class"])
class_weight_dict = dict(zip(classes, weights))

print("\n⚖️ --- Balanced Class Weights ---")
for c_id, weight in class_weight_dict.items():
    print(f"{class_names[c_id]:<45} | Weight: {weight:.4f}")

📊 --- 3-Class Distribution ---
Class 0: Stay (<= 180 days)                   | Count:  1,993 | Ratio: 2.82%
Class 1: Delayed Return (> 180 days)          | Count:    230 | Ratio: 0.33%
Class 2: True Churn (No Return)               | Count: 68,352 | Ratio: 96.85%

⚖️ --- Balanced Class Weights ---
Class 0: Stay (<= 180 days)                   | Weight: 11.8038
Class 1: Delayed Return (> 180 days)          | Weight: 102.2826
Class 2: True Churn (No Return)               | Weight: 0.3442


### ⚙️ 3. Feature Engineering

In [4]:
print("🛠️ Computing features...")

# แปลงฟิลด์ที่เป็น Decimal ให้เป็น Float
for col in ["price", "freight_value", "payment_value"]:
    if col in df.columns:
        df[col] = df[col].astype(float)

# 1. Logistics Features
df["delivery_days"] = (df["order_delivered_customer_date"] - df["order_purchase_timestamp"]).dt.days.clip(lower=0)
df["estimated_days"] = (df["order_estimated_delivery_date"] - df["order_purchase_timestamp"]).dt.days
df["delivery_vs_estimated"] = df["estimated_days"] - df["delivery_days"]

# 2. Financial
df["freight_ratio"] = np.where(df["price"] > 0, df["freight_value"] / df["price"], 0)

# 3. Review Quality
df["is_low_score"] = (df["review_score"] <= 2).astype(int)
df["is_high_score"] = (df["review_score"] == 5).astype(int)

# 4. Purchase Behaviour
df["purchase_count"] = df.groupby("customer_unique_id").cumcount() + 1
df["is_first_purchase"] = (df["purchase_count"] == 1).astype(int)
df["is_repeat_buyer"] = (df["purchase_count"] >= 2).astype(int)

# 5. Purchase Gap
df["prev_purchase_date"] = df.groupby("customer_unique_id")["order_purchase_timestamp"].shift(1)
df["days_since_last_purchase"] = (df["order_purchase_timestamp"] - df["prev_purchase_date"]).dt.days
median_gap = df.loc[df["is_repeat_buyer"] == 1, "days_since_last_purchase"].median() or 90.0
df["days_since_last_purchase"] = df["days_since_last_purchase"].fillna(median_gap)

df["avg_purchase_gap"] = df.groupby("customer_unique_id")["days_since_last_purchase"].transform("mean").fillna(median_gap)
df["gap_real"] = np.where(df["is_repeat_buyer"] == 1, df["days_since_last_purchase"], 0)
df["gap_vs_avg_real"] = np.where(df["is_repeat_buyer"] == 1, df["avg_purchase_gap"] - df["days_since_last_purchase"], 0)

# V2: First Impression -> รวมเป็น bad_experience_score (0-4)
df["first_purchase_late"] = np.where((df["is_first_purchase"] == 1) & (df["delivery_vs_estimated"] < 0), 1, 0)
df["first_purchase_bad_review"] = np.where((df["is_first_purchase"] == 1) & (df["review_score"] <= 2), 1, 0)
df["delay_days"] = (df["order_delivered_customer_date"] - df["order_estimated_delivery_date"]).dt.days.fillna(0)
df["is_extremely_late"] = (df["delay_days"] > 7).astype(int)

df["bad_experience_score"] = (
    df["is_low_score"] +
    df["first_purchase_bad_review"] +
    df["first_purchase_late"] +
    df["is_extremely_late"]
)

# จัดกลุ่ม Features (V2: ลบ 4 ฟีเจอร์ซ้ำซ้อน -> เพิ่ม bad_experience_score ตัวเดียว)
features = [
    'price', 'freight_value', 'payment_installments', 'delivery_days', 'is_first_purchase',
    'avg_purchase_gap', 'delivery_vs_estimated', 'freight_ratio', 'is_high_score',
    'gap_real', 'gap_vs_avg_real', 'bad_experience_score'
]

X = df[features].copy()
for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors="coerce")
X = X.fillna(X.median())
y = df["label_3class"]

# คำนวณ Sample Weights รายคน
sample_weights = y.map(class_weight_dict).values

print(f"✅ Features are ready! Total features: {len(features)}")
X.head(3)

🛠️ Computing features...
✅ Features are ready! Total features: 15


,price,freight_value,payment_installments,delivery_days,is_first_purchase,avg_purchase_gap,delivery_vs_estimated,freight_ratio,is_low_score,is_high_score,gap_real,gap_vs_avg_real,first_purchase_late,first_purchase_bad_review,is_extremely_late
0,69.00,17.22,8,25.0,1,12.0,2.0,0.249565,0,0,0.0,0.0,0,0,0
1,25.99,17.63,4,20.0,1,12.0,11.0,0.678338,0,0,0.0,0.0,0,0,0
2,180.00,16.89,6,13.0,1,12.0,7.0,0.093833,0,1,0.0,0.0,0,0,0


### 🤖 4. เทรนโมเดลผ่าน FLAML AutoML + K-Fold Cross Validation + MLflow Tracking
สั่งให้ระบบหาโมเดลและการตั้งค่าพารามิเตอร์ที่ดีที่สุดให้โดยอัตโนมัติภายใน 60 วินาที พร้อมส่งประวัติการสุ่มทั้งหมดขึ้นสู่หน้าเว็บย่อย (Nested Runs) ของ MLflow

In [5]:
import mlflow
mlflow.end_run()


In [6]:
import mlflow
import mlflow.data
from mlflow.data.pandas_dataset import PandasDataset
from flaml import AutoML
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import os

# 1. เชื่อมต่อ MLflow Server
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("olist_churn_3class")

# เปิดใช้งาน Autolog ของ FLAML และ LightGBM
try:
    mlflow.lightgbm.autolog(disable=True)
except:
    pass

# แบ่งข้อมูล Train/Val
X_train, X_val, y_train, y_val, w_train, w_val = train_test_split(
    X, y, sample_weights, test_size=0.2, random_state=42, stratify=y
)

# 2. ทำ Data Versioning (Dataset Tracking)
train_dataset: PandasDataset = mlflow.data.from_pandas(
    pd.concat([X_train, y_train], axis=1), 
    targets="label_3class",
    name="Olist_3Class_Balanced_Train_V2"
)

# 3. ตั้งค่าระบบ FLAML AutoML
automl = AutoML()
settings = {
    "time_budget": 60,            # จำกัดเวลาค้นหา 60 วินาที
    "metric": "macro_f1",         # คะแนนหลักที่ใช้ประเมิน
    "task": "classification",     # งานจำแนกประเภท
    "estimator_list": ["lgbm"],   # บังคับใช้ LightGBM
    "n_splits": 5,                # ทำ 5-Fold Cross Validation
    "seed": 42,
    "verbose": 1,
    "mlflow_logging": False
}

with mlflow.start_run(run_name="flaml_automl_v2_bad_exp_score") as run:
    print("🚀 Starting FLAML AutoML Parameter Search (Model V2)...\n")
    
    # ล็อกข้อมูลเวอร์ชั่นต้นน้ำ
    mlflow.log_input(train_dataset, context="training")
    
    # เทรนหาโมเดลที่ดีที่สุดพร้อมส่งค่าน้ำหนักแก้ Imbalance เข้าไปประมวลผล
    automl.fit(
        X_train=X_train, 
        y_train=y_train, 
        sample_weight=w_train, 
        **settings
    )
    
    # ดึงโมเดลที่ดีที่สุดที่ได้จากค้นหา
    best_model = automl.model.estimator
    
    # ทำนายผลบน Validation Set
    y_pred = best_model.predict(X_val)
    
    # คำนวณคะแนนหลัก
    acc = accuracy_score(y_val, y_pred)
    macro_f1 = f1_score(y_val, y_pred, average="macro")
    
    print("\n⭐ best model found via AutoML:")
    print(f"Best Config: {automl.best_config}")
    print(f"Accuracy on validation set: {acc:.4f}")
    print(f"Macro F1 on validation set: {macro_f1:.4f}")
    print("\n--- Detailed Classification Report ---")
    print(classification_report(y_val, y_pred, target_names=[class_names[i] for i in range(3)]))
    
    # ล็อกค่า AutoML parameters หลัก
    mlflow.log_params(automl.best_config)
    mlflow.log_params({
        "class_weight_class_0": class_weight_dict[0],
        "class_weight_class_1": class_weight_dict[1],
        "class_weight_class_2": class_weight_dict[2],
        "model_version": "v2",
        "feature_change": "merged_4_to_bad_experience_score"
    })
    mlflow.log_metric("val_accuracy", acc)
    mlflow.log_metric("val_macro_f1", macro_f1)
    
    # 4. วาดและบันทึกรูปกราฟ Confusion Matrix เข้าสู่ระบบ Artifact
    plt.figure(figsize=(8, 6))
    cm = confusion_matrix(y_val, y_pred)
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=[class_names[i] for i in range(3)],
        yticklabels=[class_names[i] for i in range(3)]
    )
    plt.title("Confusion Matrix - AutoML Best Model V2")
    plt.ylabel("Actual Class")
    plt.xlabel("Predicted Class")
    plt.tight_layout()
    
    cm_path = "confusion_matrix_automl.png"
    plt.savefig(cm_path)
    plt.close()
    mlflow.log_artifact(cm_path)
    if os.path.exists(cm_path): os.remove(cm_path)

    # 5. วาดและบันทึกรูปกราฟ Feature Importance
    importances = best_model.feature_importances_
    feat_imp_df = pd.DataFrame({
        'Feature': X_train.columns,
        'Importance': importances
    }).sort_values(by='Importance', ascending=True)

    plt.figure(figsize=(10, 6))
    plt.barh(feat_imp_df['Feature'], feat_imp_df['Importance'], color='skyblue')
    plt.title('Feature Importance - AutoML Best Model V2')
    plt.xlabel('Importance Score')
    plt.tight_layout()

    fi_path = "feature_importance_automl.png"
    plt.savefig(fi_path)
    plt.close()
    mlflow.log_artifact(fi_path)
    if os.path.exists(fi_path): os.remove(fi_path)
    
    mlflow.sklearn.log_model(best_model, artifact_path="model", serialization_format="cloudpickle")
    
    print(f"\n🎉 AutoML K-Fold Run Logged! Run ID: {run.info.run_id}")

🚀 Starting FLAML AutoML Parameter Search...

⭐ best model found via AutoML:
Best Config: {'n_estimators': 27, 'num_leaves': 159, 'min_child_samples': 4, 'learning_rate': np.float64(0.006971638112513486), 'log_max_bin': 5, 'colsample_bytree': np.float64(0.8533693238593487), 'reg_alpha': np.float64(0.015592522706381657), 'reg_lambda': np.float64(4.057693397324225)}
Accuracy on validation set: 0.9895
Macro F1 on validation set: 0.9087

--- Detailed Classification Report ---
                                      precision    recall  f1-score   support

         Class 0: Stay (<= 180 days)       0.74      0.98      0.84       399
Class 1: Delayed Return (> 180 days)       0.84      0.93      0.89        46
     Class 2: True Churn (No Return)       1.00      0.99      0.99     13670

                            accuracy                           0.99     14115
                           macro avg       0.86      0.97      0.91     14115
                        weighted avg       0.99      0

2026/07/13 17:44:01 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



🎉 AutoML K-Fold Run Logged! Run ID: 2b1d3e0f25674f7aab60e191f70bb2f8
🏃 View run flaml_automl_kfold_run at: http://127.0.0.1:5000/#/experiments/2/runs/2b1d3e0f25674f7aab60e191f70bb2f8
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


### 🔍 5. คำนวณ SHAP Analysis & บันทึกคำอธิบายฟีเจอร์ลงระบบ
คำนวณหาเหตุผลเบื้องหลังการทำนายของฟีเจอร์หลักทั้ง 12 ตัว (V2) และส่งออกกราฟ Summary Beeswarm Plot เข้าสู่ระบบเพื่อตรวจสอบผ่านหน้าเว็บ

In [10]:
import shap
import mlflow.shap

print("🔮 Calculating SHAP values (it may take a moment)...")

# สร้าง Explainer ด้วยโมเดลและข้อมูลสุ่มกลุ่มตัวอย่าง 200 แถวเพื่อให้คำนวณได้รวดเร็ว
sample_X = X_val.sample(n=200, random_state=42) if len(X_val) > 200 else X_val
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(sample_X)

# สร้างและบันทึก Beeswarm Plot ของ SHAP ลงเครื่องคอมเพื่อส่งเข้า MLflow
# หมายเหตุ: สำหรับ Multiclass โมเดล SHAP จะแยกเป็น 3 คลาสตามลำดับ
plt.figure(figsize=(10, 6))
# อ้างอิง Class 2 (True Churn) เป็นเกณฑ์วิเคราะห์ เพื่อดูว่าฟีเจอร์ไหนส่งผลให้ลูกค้าหายมากที่สุด
# ตรวจสอบมิติและเลือก slice ดัชนีแกนสุดท้าย [..., 2] ให้เหมาะสมกับโมเดล 3 คลาส
shap_values_class2 = shap_values[..., 2] if shap_values.ndim == 3 else shap_values[2]

# วาด Beeswarm Plot
shap.summary_plot(shap_values_class2, sample_X, show=False)

plt.title("SHAP Feature Impact - Class 2: True Churn")
plt.tight_layout()

shap_path = "shap_beeswarm.png"
plt.savefig(shap_path)
plt.close()

# เปิดเซสชั่นอัปเดตไฟล์เข้าไปยังการรันล่าสุดของ MLflow
last_run_id = mlflow.last_active_run().info.run_id
with mlflow.start_run(run_id=last_run_id):
    # 1. บันทึกรูปภาพกราฟ Summary
    mlflow.log_artifact(shap_path)
    
    # 2. บันทึกโมเดลคำอธิบายผลลัพธ์แบบ Interactive HTML ลงในระบบ MLflow Directly
        # 🌟 รูปแบบความคุ้มกันในกล่องที่ 5:
    try:
        mlflow.shap.log_explanation(best_model.predict, sample_X)
    except Exception as e:
        print(f"⚠️ Warning: Could not log interactive SHAP explanation due to: {e}. However, summary beeswarm plot was successfully logged as an artifact.")

    
    print("✅ SHAP analysis logged successfully under current run!")

if os.path.exists(shap_path): os.remove(shap_path)

🔮 Calculating SHAP values (it may take a moment)...
⚠️ Warning: Could not log interactive SHAP explanation due to: property 'feature_names_in_' of 'LGBMClassifier' object has no setter. However, summary beeswarm plot was successfully logged as an artifact.
✅ SHAP analysis logged successfully under current run!
🏃 View run flaml_automl_kfold_run at: http://127.0.0.1:5000/#/experiments/2/runs/2b1d3e0f25674f7aab60e191f70bb2f8
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


### 🌐 6. ขึ้นทะเบียนโมเดล (Model Registry)
บันทึกรหัสโมเดลนี้ไว้ในระบบส่วนกลางเพื่อให้ระบบทำนายผลและแดชบอร์ดปลายทางดึงไปใช้ทำพยากรณ์จริงต่อทันที

In [ ]:
# ทำการลงทะเบียนโมเดลที่ดีที่สุด
model_uri = f"runs:/{last_run_id}/model"
model_name = "Olist_3Class_MLOps_Model"

print(f"📦 Registering model '{model_name}'...")
registered_model = mlflow.register_model(model_uri, model_name)
print(f"🎉 Model registered successfully! Version: {registered_model.version}")